<a href="https://colab.research.google.com/github/GilliardMorandim/mba-tcc-usp-inadimplencia/blob/eda%2Ffeature/pre_processing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

SEMANA 2 – 9 a 15 de Dezembro

Objetivo: Pré-processamento completo
Tarefas:

Normalização (z-score)

Tratamento de missings (MICE/KNN)

One-hot encoding / target encoding

Preparar dataset final para modelagem

Entrega:
Dataset “master” pré-processado.

- incluir a pct_do_contrato no parcelas_main para saber se é uma parcela do começo do contrato ou não
-

# Requirements

In [1]:
!apt-get install openjdk-11-jdk-headless -qq > /dev/null
!pip install pyspark


In [2]:
import matplotlib.pyplot as plt
from google.colab import drive
import os
import pandas as pd

drive.mount('/content/drive')


Mounted at /content/drive


# Lendo Arquivo Pyspark

In [3]:
from pyspark.sql import SparkSession

spark = (SparkSession.builder
         .appName("LoadAllCSVs")
         .config("spark.driver.memory", "8g")
         .config("spark.executor.memory", "8g")
         .config("spark.sql.files.maxPartitionBytes", "256m")
         .getOrCreate())

print("Spark iniciado!")

Spark iniciado!


In [4]:
dfs_spark = {}

base_path = "/content/drive/MyDrive/MBA - Ciencia de Dados - USP/dados_tcc/"

files = [f for f in os.listdir(base_path) if f.endswith(".csv")]

for file in files:
    full_path = os.path.join(base_path, file)
    print(f"\n📥 Lendo via PySpark: {file}")

    try:
        df = (spark.read
              .option("header", "true")
              .option("inferSchema", "true")
              .csv(full_path))

        key = file.replace(".csv", "")
        dfs_spark[key] = df

        print(f"✔ OK - Linhas (estimado pela Spark): {df.count()} | Colunas: {len(df.columns)}")

    except Exception as e:
        print(f"❌ Erro ao ler {file}: {e}")

print("\nTodos os arquivos foram processados!")

globals().update(dfs_spark)


📥 Lendo via PySpark: pre_aprovado.csv
✔ OK - Linhas (estimado pela Spark): 42171363 | Colunas: 11

📥 Lendo via PySpark: parcelas.csv
✔ OK - Linhas (estimado pela Spark): 1865444 | Colunas: 21

📥 Lendo via PySpark: contratos.csv
✔ OK - Linhas (estimado pela Spark): 382539 | Colunas: 26

📥 Lendo via PySpark: score_credito.csv
✔ OK - Linhas (estimado pela Spark): 266126 | Colunas: 3

📥 Lendo via PySpark: analise_conversao.csv
✔ OK - Linhas (estimado pela Spark): 538023 | Colunas: 12

📥 Lendo via PySpark: analise_conversao_v2.csv
✔ OK - Linhas (estimado pela Spark): 538023 | Colunas: 12

📥 Lendo via PySpark: analise_conversao_v3.csv
✔ OK - Linhas (estimado pela Spark): 533525 | Colunas: 11

Todos os arquivos foram processados!


# Pre-Processing

In [9]:
parcelas_main = parcelas

contratos_keys = contratos.select(
    "id_contrato",
    "uuid_cliente",
    "id_contrato_original",
    "id_contrato_pai",
    "cpf_hash_sha256").filter("id_contrato IS NOT NULL")

#parcelas_main = parcelas_main.join(
 #   contratos_keys,
  #  parcelas_main.id_contrato == contratos_keys.id_contrato,
   # "left")

parcelas_main = parcelas.join(
    contratos_keys,
    on="id_contrato",
    how="left"
)

parcelas_main = parcelas_main.filter("data_vencimento < '2025-11-01'")

#Flag inadimplência

In [10]:
from pyspark.sql.functions import col, when

parcelas_main = parcelas_main.withColumn(
    "flag_inadimplencia_30_days",
    when(col("qtd_dias_de_atraso")>30,1).otherwise(0)).withColumn(
     "flag_inadimplencia_90_days",when(col("qtd_dias_de_atraso")>90,1).otherwise(0))

#parcelas_main.show(5, truncate=False)

+--------------------------------------+--------------+---------------+------+---------+------------------------------+------------+-------------+------------+-----------+-----------+-----------------+----------+--------------+------------------+--------------------------+----------------+--------------------------------------+-------+------------------+-------------------------------+--------------------------------------+--------------------------------------+---------------+----------------------------------------------------------------+--------------------------+--------------------------+
|id_contrato                           |data_pagamento|data_vencimento|valor |valor_iof|valor_financiado_principal_iof|qtd_parcelas|valor_parcela|valor_tarifa|valor_juros|valor_iof_2|valor_amortizacao|valor_pago|valor_desconto|valor_juros_atraso|valor_juros_remuneratorios|status_pagamento|id_parcela                            |parcela|qtd_dias_de_atraso|contract_installments_status_id|uuid_clien

# Normalização da parcela

In [11]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# janela por contrato
w = Window.partitionBy("id_contrato")

parcelas_main = (
    parcelas_main

    .withColumn("min_parcela", F.min("parcela").over(w))
    .withColumn("max_parcela", F.max("parcela").over(w))
    .withColumn(
        "parcela_norm_0_1",
        (F.col("parcela") - F.col("min_parcela")) /
        (F.col("max_parcela") - F.col("min_parcela"))
    )
    .drop("min_parcela", "max_parcela")
)

parcelas_main.orderBy("data_vencimento").filter(F.col("id_contrato") == "{B00ADF21-0E77-4C6B-9C10-A8A51B8C04BC}").show(100, truncate=False)
#parcelas_main.show(10,truncate=False)

+--------------------------------------+--------------+---------------+------+---------+------------------------------+------------+-------------+------------+-----------+-----------+-----------------+----------+--------------+------------------+--------------------------+----------------+--------------------------------------+-------+------------------+-------------------------------+--------------------------------------+--------------------------------------+---------------+----------------------------------------------------------------+--------------------------+--------------------------+-------------------+
|id_contrato                           |data_pagamento|data_vencimento|valor |valor_iof|valor_financiado_principal_iof|qtd_parcelas|valor_parcela|valor_tarifa|valor_juros|valor_iof_2|valor_amortizacao|valor_pago|valor_desconto|valor_juros_atraso|valor_juros_remuneratorios|status_pagamento|id_parcela                            |parcela|qtd_dias_de_atraso|contract_installments_